
## MNIST & Fashion-MNIST
### anchor 直交 #→非直交

In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np

def data_setup(DATA):
    if DATA == "MNIST":
        # Load MNIST from Keras (60k train, 10k test)
        from tensorflow.keras.datasets import mnist
        (X_train, y_train), (X_test, y_test) = mnist.load_data()

    elif DATA == "FashionMNIST":
        # Load Fashion-MNIST from Keras (60k train, 10k test)
        from tensorflow.keras.datasets import fashion_mnist
        (X_train, y_train), (X_test, y_test) = fashion_mnist.load_data()

    else:
        raise ValueError("Unsupported dataset. Please choose 'MNIST' or 'FashionMNIST'.")

    # Combine into a single 70k dataset (to mimic the OpenML version)
    X = np.concatenate([X_train, X_test], axis=0)
    y = np.concatenate([y_train, y_test], axis=0)

    # Flatten 28x28 images to 784-dimensional vectors and rescale to [0, 1]
    X = X.reshape(-1, 28 * 28).astype(np.float32) / 255.0
    y = y.astype(int)

    # Split into train/test: 10k for test (like your original code)
    train_data_all, test_data_all, train_label_all, test_label_all = train_test_split(
        X, y, test_size=10000, random_state=42
    )

    return train_data_all, train_label_all, test_data_all, test_label_all

In [ ]:
def data_splitting(train_data_all, train_label_all, test_data_all, test_label_all, NUM_USERS, INDIVIDUAL_SAMPLES):

    train_sample_indices = np.random.choice(train_data_all.shape[0], NUM_USERS*INDIVIDUAL_SAMPLES, replace=False)
    X_train = train_data_all[train_sample_indices]
    y_train = train_label_all[train_sample_indices]

    # Random subset of the test data (1000 samples)
    test_sample_indices = np.random.choice(test_data_all.shape[0], 1000, replace=False)
    X_test = test_data_all[test_sample_indices]
    y_test = test_label_all[test_sample_indices]

    # Randomly split the data into NUM_USERS subsets with INDIVIDUAL_SAMPLES samples each
    X_train_list = np.array_split(X_train, NUM_USERS)
    y_train_list = np.array_split(y_train, NUM_USERS)

    return X_train, y_train, X_train_list, y_train_list, X_test, y_test

In [ ]:
from scipy.stats import ortho_group
from sklearn.utils.extmath import randomized_svd

def basis_selection(m, l, c, SAME_SPAN=True, ORTHONORMAL=True, X_train_list = None):


    if X_train_list:

        if SAME_SPAN:

            _, _, Vt = randomized_svd(X_train_list[0], n_components=l)
            Q = Vt.T

            if ORTHONORMAL:

                basis_list = [Q @ ortho_group.rvs(l) for _ in range(c)]

            else:

                basis_list = [Q @ np.random.uniform(size=(l, l)) for _ in range(c)]

        else:

            basis_list = []
            for i in range(c):
                _, _, Vt = randomized_svd(X_train_list[i], n_components=l)

                if ORTHONORMAL:
                    basis_list.append(Vt.T @ ortho_group.rvs(l))

                else:
                    basis_list.append(Vt.T @ np.random.uniform(size=(l, l)))

    else:

        if SAME_SPAN:

            common_subspace = np.random.uniform(size=(m, l))

            if ORTHONORMAL:

                Q, _ = np.linalg.qr(common_subspace)
                basis_list = [Q @ ortho_group.rvs(l) for _ in range(c)]

            else:

                basis_list = [common_subspace @ np.random.uniform(size=(l, l)) for _ in range(c)]

        else:

            if ORTHONORMAL:

                basis_list = [np.linalg.qr(np.random.uniform(size=(m, l)))[0] for _ in range(c)]

            else:

                basis_list = [np.random.uniform(size=(m, l)) for _ in range(c)]


    return basis_list


In [ ]:
from __future__ import annotations

from typing import Sequence, List, Tuple

import numpy as np
from scipy.linalg import qr, solve_triangular
from scipy.sparse.linalg import LinearOperator, svds
from scipy.stats import ortho_group


# -----------------------------------------------------------------------------#
#  Generic helpers                                                              #
# -----------------------------------------------------------------------------#
def _validate_inputs(Ai_list: Sequence[np.ndarray]) -> tuple[int, int, int, np.dtype]:
    if not Ai_list:
        raise ValueError("Ai_list is empty")

    a, l = Ai_list[0].shape
    if a < l:
        raise ValueError("Each A_i must be tall (a ≥ ℓ)")

    for k, A in enumerate(Ai_list, start=1):
        if A.shape != (a, l):
            raise ValueError(f"A_{k} has shape {A.shape}, expected {(a, l)}")

    return a, l, len(Ai_list), Ai_list[0].dtype


def _svds_descending(op: LinearOperator,
                     k: int,
                     random_state: int | None) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Truncated SVD via ARPACK that *accepts LinearOperator*.
    Returns (U, Σ, Vᵀ) with singular values sorted in descending order.
    """
    u, s, vt = svds(op, k=k, which='LM',
                    return_singular_vectors=True,
                    random_state=random_state)

    order = np.argsort(s)[::-1]
    return u[:, order], s[order], vt[order, :]


def _qr_r_factors(Ai_list: Sequence[np.ndarray]) -> List[np.ndarray]:
    """Economy‑QR of each Aᵢ → list of Rᵢ factors."""
    return [qr(A, mode="economic")[1] for A in Ai_list]


# -----------------------------------------------------------------------------#
#  Main entry point                                                             #
# -----------------------------------------------------------------------------#
def basis_alignment(Ai_list: Sequence[np.ndarray],
                    method: str = "Imakura",
                    *,
                    random_state: int | None = None) -> np.ndarray:
    """
    Compute the (ℓ×ℓ) change‑of‑basis matrices for several tall matrices
    according to one of three published methods.

    Parameters
    ----------
    Ai_list : sequence of ndarray
        List ``[A₀, …, A_{c‑1}]`` with identical shape (a, ℓ),  a > ℓ.
    method : {"Imakura", "Kawakami", "Nosaka"}, default "Imakura"
        Choice of algorithm.
    random_state : int or None, optional
        Seed for the RNG (used by ARPACK’s starting vector and Nosaka’s
        orthogonal draw).

    Returns
    -------
    changeofbasis : ndarray, shape (c, ℓ, ℓ)
    """
    a, l, c, dtype = _validate_inputs(Ai_list)
    method_key = method.casefold()

    if method_key not in {"imakura", "kawakami", "nosaka", "imakura_rand", "imakura_rand_orth", "kawakami_rand_orth", "nosaka_rand"}:
        raise ValueError("method must be 'Imakura', 'Kawakami' or 'Nosaka'")

    rng = np.random.default_rng(random_state)
    changeofbasis = np.empty((c, l, l), dtype=dtype)

    # ====================================================================== #
    #  1.  I M A K U R A                                                     #
    # ====================================================================== #
    if method_key in ["imakura","imakura_rand","imakura_rand_orth"]:

        # --- LinearOperator H = [A₀ | … | A_{c−1}] -------------------------
        def _H_matvec(x: np.ndarray) -> np.ndarray:           # x (cℓ,)
            x = x.astype(dtype, copy=False).reshape(c, l)
            y = np.zeros(a, dtype=dtype)
            for A, xi in zip(Ai_list, x):
                y += A @ xi
            return y

        def _H_rmatvec(y: np.ndarray) -> np.ndarray:         # y (a,)
            return np.concatenate([A.T @ y for A in Ai_list])

        H_op = LinearOperator((a, c * l),
                              matvec=_H_matvec,
                              rmatvec=_H_rmatvec,
                              dtype=dtype)

        # --- left singular vectors  U  via svds ----------------------------
        U, _, _ = _svds_descending(H_op, k=l, random_state=random_state)

        #if method_key == "imakura_rand":
        #    U = U @ np.random.uniform(size=(l, l))

        if method_key == "imakura_rand":
            U = U @ np.random.uniform(size=(l, l))

        if method_key == "imakura_rand_orth":
            U = U @ ortho_group.rvs(l, random_state=rng)

        # --- slice‑wise change‑of‑basis -----------------------------------
        for i, A in enumerate(Ai_list):
            Q, R = qr(A, mode="economic")
            Y = Q.T @ U                           # (ℓ, ℓ)
            changeofbasis[i] = solve_triangular(R, Y, lower=False)




    # ====================================================================== #
    #  2.  K A W A K A M I                                                   #
    # ====================================================================== #
    elif method_key in ["kawakami", "kawakami_rand_orth"]:

        R_list = _qr_r_factors(Ai_list)           # Rᵢ once, reused

        # --- LinearOperator W = [Q₀ | … | Q_{c−1}] -------------------------
        def _W_matvec(x: np.ndarray) -> np.ndarray:
            x = x.astype(dtype, copy=False).reshape(c, l)
            y = np.zeros(a, dtype=dtype)
            for A, R, xi in zip(Ai_list, R_list, x):
                y += A @ solve_triangular(R, xi, lower=False)
            return y

        def _W_rmatvec(y: np.ndarray) -> np.ndarray:
            blocks = []
            for A, R in zip(Ai_list, R_list):
                z = A.T @ y
                blocks.append(solve_triangular(R.T, z, lower=True))
            return np.concatenate(blocks)

        W_op = LinearOperator((a, c * l),
                              matvec=_W_matvec,
                              rmatvec=_W_rmatvec,
                              dtype=dtype)

        # --- right singular vectors Vᵀ via svds ----------------------------
        _, _, Vt = _svds_descending(W_op, k=l, random_state=random_state)

        # split Vᵀ into c blocks of (ℓ×ℓ)
        Vt_blocks = Vt.reshape(l, c, l).transpose(1, 0, 2)   # (c, ℓ, ℓ)

        U = np.eye(l)
        if method_key == "kawakami_rand_orth":
            U = ortho_group.rvs(l, random_state=rng)

        for i, (R, Vb) in enumerate(zip(R_list, Vt_blocks)):
            changeofbasis[i] = solve_triangular(R, Vb.T @ U, lower=False)
            changeofbasis[i] *= np.sqrt(c)

    # ====================================================================== #
    #  3.  N O S A K A                                                       #
    # ====================================================================== #
    elif method_key in ["nosaka","nosaka_rand"]:

        if method_key == "nosaka_rand":
            Z =  Ai_list[0] @ ortho_group.rvs(l, random_state=rng)

        else:
            Z = Ai_list[0]

        A_big = np.stack(Ai_list)                              # (c, a, ℓ)
        M = np.einsum('ial,aj->ilj', A_big, Z, optimize=True)  # (c, ℓ, ℓ)

        U, _, Vt = np.linalg.svd(M, full_matrices=False)
        changeofbasis = U @ Vt                                # (c, ℓ, ℓ)
    changeofbasis *= np.sqrt(a)
    return changeofbasis

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

def model_acc(train_data, test_data, train_labels, test_labels, verbose=False, model="mlp"):

    if model=="svm":

        # Define the SVM model
        model = SVC(random_state=1, verbose=verbose)

        # Train (fit) the model
        model.fit(train_data, train_labels)
        # Predict and compute accuracy
        y_pred = model.predict(test_data)
        test_acc = accuracy_score(test_labels, y_pred)
        if verbose:
            print(f"Test accuracy (SVM): {test_acc:.3f}")

        return test_acc

    elif model=="mlp":

        # Define the MLPClassifier (very simple feedforward neural network)
        # Using a single hidden layer with 256 neurons
        mlp = MLPClassifier(
            hidden_layer_sizes=(256,),
            activation='relu',
            solver='adam',
            batch_size=32,
            random_state=1,
            max_iter=1000,  # Increase max_iter for better convergence
            early_stopping=True,  # Enable early stopping to prevent overfitting
            verbose=verbose
        )

        # Train (fit) the model
        mlp.fit(train_data, train_labels)

        # Predict and compute accuracy
        y_pred = mlp.predict(test_data)
        test_acc = accuracy_score(test_labels, y_pred)

        if verbose:
            print(f"Test accuracy (MLP): {test_acc:.3f}")

        return test_acc

    elif model=="rf":

        # Define the Random Forest model
        model = RandomForestClassifier(random_state=1, verbose=verbose)

        # Train (fit) the model
        model.fit(train_data, train_labels)
        # Predict and compute accuracy
        y_pred = model.predict(test_data)
        test_acc = accuracy_score(test_labels, y_pred)
        if verbose:
            print(f"Test accuracy (Random Forest): {test_acc:.3f}")

        return test_acc

    else:
        raise ValueError("Unsupported model. Please choose 'mlp' or 'svm' or 'rf.")

In [ ]:
def compute_accuracy(X_train_list, X_test, y_train_list, y_test, bases, cobs, model="mlp", random_indice=None):

    c = len(X_train_list)
    """
    Compute the accuracy of the model using the transformed data.
    """
    X_train_hat = np.vstack([X_train_list [i] @ bases[i] @ cobs[i] for i in range(c)])
    X_test_hat_list = [X_test @ bases[i] @ cobs[i] for i in range(c)]

    if random_indice:
        return model_acc(X_train_hat, X_test_hat_list[random_indice], np.hstack(y_train_list), y_test, verbose=False, model=model)

    else:
        accs = []
        for i in range(c):
            acc = model_acc(X_train_hat, X_test_hat_list[i], np.hstack(y_train_list), y_test, verbose=False, model=model)
            accs.append(acc)
        return np.mean(accs)

In [ ]:
from re import A
# experiment.py
from __future__ import annotations

import os
import random
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd

# ---------------------------------------------------------------------
# 0. Verify that the helper functions you rely on really exist.
#    This fails fast instead of producing a cryptic NameError minutes later.
# ---------------------------------------------------------------------
REQUIRED_FUNCS = [
    "data_setup", "data_splitting", "model_acc",
    "basis_selection", "basis_alignment", "compute_accuracy"
]
missing = [f for f in REQUIRED_FUNCS if f not in globals()]
if missing:                                                             # noqa: WPS507
    raise ImportError(
        f"Missing helper function(s): {', '.join(missing)}. "
        "Import or define them before calling experiment_setup()."
    )

# ---------------------------------------------------------------------
# 1. Main entry point
# ---------------------------------------------------------------------
def experiment_setup(
    DATA: str,
    c: int,
    INDIVIDUAL_SAMPLES: int,
    l: int,
    a: int,
    *,
    iterations: int = 10,
    save: bool = True,
    out_dir: Path | str = Path(f"/content"),
    seed: int | None = 42,
) -> None:
    """
    Run `iterations` experiments and persist summary CSVs.

    Parameters
    ----------
    DATA :
        Name of the dataset (used in output-file names).
    c :
        Number of clients / users.
    INDIVIDUAL_SAMPLES :
        Samples per client.
    l :
        Anchor dimension *l* in your notation.
    a :
        Number of anchor rows to sample.
    iterations :
        Repetition count (default 10).
    save :
        Whether to append results to CSV (default True).
    out_dir :
        Directory where CSVs are stored
    seed :
        Global RNG seed for reproducibility.
    """
    # -----------------------------------------------------------------
    # 1.1  Global setup
    # -----------------------------------------------------------------
    if seed is not None:
        random.seed(seed)
        np.random.seed(seed)

    out_path = Path(out_dir).expanduser()
    out_path.mkdir(parents=True, exist_ok=True)

    train_X_all, train_y_all, test_X_all, test_y_all = data_setup(DATA)
    m: int = train_X_all.shape[1]

    # -----------------------------------------------------------------
    # 1.2  Prepare the four (SAME_SPAN, ORTHONORMAL) cases once
    # -----------------------------------------------------------------
    CASES: List[Tuple[bool, bool, str]] = [
        (False, False, "Diffspan"),
        (True,  True,  "SamespanOrth"),
        (True,  False, "Samespan"),
        (False, True,  "DiffspanOrth"),
    ]

    # -----------------------------------------------------------------
    # 1.3  Iterate
    # -----------------------------------------------------------------


    # 1.3.1  Split data for this iteration
    (
        X_train, y_train,
        X_train_list, y_train_list,
        X_test,  y_test
    ) = data_splitting(
        train_X_all, train_y_all,
        test_X_all,  test_y_all,
        c, INDIVIDUAL_SAMPLES
    )

    random_indice = random.randrange(c)
    # 1.3.2  Central & first-user baselines
    #acc_central_svm = model_acc(X_train,     X_test,  y_train,     y_test, model="svm")
    #acc_central_mlp = model_acc(X_train,     X_test,  y_train,     y_test, model="mlp")
    #acc_local_svm   = model_acc(X_train_list[random_indice], X_test, y_train_list[random_indice], y_test, model="svm")
    #acc_local_mlp   = model_acc(X_train_list[random_indice], X_test, y_train_list[random_indice], y_test, model="mlp")

    # 1.3.3  Anchor matrix shared across the four cases
    anchor = np.random.uniform(size=(a, m))
    # Apply QR decomposition to make the columns of 'anchor' orthogonal as requested by the user.
    # Since a > m (1000 > 784), qr(anchor) will return a Q matrix of shape (a, m) with orthogonal columns.
    #Q, _ = np.linalg.qr(anchor)
    #anchor = Q

    # 1.3.4  Loop over SAME_SPAN/ORTHONORMAL combinations
    for same_span, orthonorm, tag in CASES:

        bases = basis_selection(m, l, c,
                                SAME_SPAN=same_span,
                                ORTHONORMAL=orthonorm,
                                X_train_list=X_train_list)

        anchor_views = [anchor @ basis for basis in bases]

        for it in range(iterations):

            cobs = {
                method: basis_alignment(anchor_views, method=method, random_state=it)
                for method in ("Imakura", "Nosaka", "Nosaka_rand") # "Imakura", "Imakura_rand", "Imakura_rand_orth", "Kawakami", "Kawakami_rand_orth",
            }

            # 1.3.5  Compute accuracies for three methods × two models
            results: Dict[str, float] = {}
            #results: Dict[str, float] = {
            #    "acc_central_svm": acc_central_svm,
            #    "acc_central_mlp": acc_central_mlp,
            #    "acc_local_svm":   acc_local_svm,
            #    "acc_local_mlp":   acc_local_mlp,
            #}

            for method in ("Imakura", "Nosaka", "Nosaka_rand"): # "Imakura", "Imakura_rand", "Imakura_rand_orth", "Kawakami", "Kawakami_rand_orth",
                results[f"acc_{method}_svm"] = compute_accuracy(
                    X_train_list, X_test, y_train_list, y_test,
                    bases, cobs[method], model="svm", random_indice=random_indice
                )
                results[f"acc_{method}_mlp"] = compute_accuracy(
                    X_train_list, X_test, y_train_list, y_test,
                    bases, cobs[method], model="mlp", random_indice=random_indice
                )

            # 1.3.6  Persist and echo
            _log_results(results, DATA, tag, out_path, save)

            # ---- Console feedback (one line) -------------------------
            print(
                f"[{it + 1:02}/{iterations}] {tag:<12} "
                + ", ".join(f"{k}={v:.4f}" for k, v in results.items())
            )

    print(f"✅ All {iterations} iteration(s) completed.")


# ---------------------------------------------------------------------
# 2. Helper: append a one-row DataFrame to <DATA>_<tag>.csv
# ---------------------------------------------------------------------
def _log_results(
    row_dict: Dict[str, float],
    data_name: str,
    tag: str,
    out_dir: Path,
    save: bool,
) -> None:
    """Append one result row to the appropriate CSV file."""
    if not save:
        return
    fname = out_dir / f"{data_name}_{tag}.csv"
    df = pd.DataFrame([row_dict])
    df.to_csv(fname, mode="a", header=not fname.exists(), index=False)


In [ ]:
#experiment_setup("MNIST", c=20, INDIVIDUAL_SAMPLES=100, l=10, a=1000, iterations=100)
experiment_setup("FashionMNIST", c=20, INDIVIDUAL_SAMPLES=100, l=10, a=1000, iterations=100)

[01/100] Diffspan     acc_Imakura_svm=0.7580, acc_Imakura_mlp=0.7630, acc_Nosaka_svm=0.5610, acc_Nosaka_mlp=0.4240, acc_Nosaka_rand_svm=0.5640, acc_Nosaka_rand_mlp=0.6130
[02/100] Diffspan     acc_Imakura_svm=0.7580, acc_Imakura_mlp=0.7560, acc_Nosaka_svm=0.5610, acc_Nosaka_mlp=0.4240, acc_Nosaka_rand_svm=0.5660, acc_Nosaka_rand_mlp=0.5350
[03/100] Diffspan     acc_Imakura_svm=0.7580, acc_Imakura_mlp=0.7640, acc_Nosaka_svm=0.5610, acc_Nosaka_mlp=0.4240, acc_Nosaka_rand_svm=0.5640, acc_Nosaka_rand_mlp=0.4450
[04/100] Diffspan     acc_Imakura_svm=0.7580, acc_Imakura_mlp=0.7660, acc_Nosaka_svm=0.5610, acc_Nosaka_mlp=0.4240, acc_Nosaka_rand_svm=0.5650, acc_Nosaka_rand_mlp=0.5300
[05/100] Diffspan     acc_Imakura_svm=0.7580, acc_Imakura_mlp=0.7580, acc_Nosaka_svm=0.5610, acc_Nosaka_mlp=0.4240, acc_Nosaka_rand_svm=0.5640, acc_Nosaka_rand_mlp=0.5880
[06/100] Diffspan     acc_Imakura_svm=0.7580, acc_Imakura_mlp=0.7590, acc_Nosaka_svm=0.5610, acc_Nosaka_mlp=0.4240, acc_Nosaka_rand_svm=0.5640, a

KeyboardInterrupt: 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')